# Shooting cost evaluator

> Compose a world model (`Dynamics`) and an `Objective` into the `Costable` solvers optimise.

In [ ]:
#| default_exp planning.evaluator

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from typing import Callable

import torch

from stable_marl.protocols import Dynamics, Objective

In [ ]:
#| export
def flat_goal_encode(model: Dynamics, info_dict: dict, goal_obs_key: str = 'pixels') -> torch.Tensor:
    """
    Encode the goal for models whose latent is one tensor: the goal observation (`goal`)
    takes the place of `goal_obs_key` (each agent's `pixels` by default), and every
    `goal_<key>` the place of `<key>` (e.g. `goal_position` -> `position`); inputs with
    no goal value (e.g. `direction`) keep their latest entries.
    Pair with `GoalMSE`.
    """
    assert 'goal' in info_dict, "goal not in info_dict"
    goal = {k: v[:, 0] for k, v in info_dict.items() if torch.is_tensor(v)}   # drop the candidate axis
    goal[goal_obs_key] = goal['goal']
    for k in list(goal):
        if k.startswith('goal_'):
            goal[k[len('goal_'):]] = goal.pop(k)
    goal.pop('action', None)
    goal.pop('action_history', None)   # past actions are context, not goal
    T = goal[goal_obs_key].shape[1]
    for k, v in goal.items():          # inputs with no goal value (e.g. direction): their latest T entries
        if v.ndim > 1 and v.shape[1] > T:
            goal[k] = v[:, -T:]
    return model.encode(goal)['emb']


def split_goal_encode(model: Dynamics, info_dict: dict) -> torch.Tensor:
    """
    Encode the goal for models whose latent concatenates several sources (stable-worldmodel's
    `PreJEPA`): the pixel embedding and one embedding per extra encoder. The action encoder is
    left out (a goal prescribes a state, not an action), and the cost has to compare the parts,
    so the per-source goal embeddings are stored in `info_dict` (`pixels_goal_emb`,
    `<key>_goal_emb`) next to the fused one, which is returned (for `goal_emb`). Score them
    with one `GoalMSE` per source, combined with `WeightedSum`.
    """
    assert 'goal' in info_dict, "goal not in info_dict"
    emb_keys = [k for k in model.extra_encoders if k != 'action']
    goal = {k: v[:, 0] for k, v in info_dict.items() if torch.is_tensor(v)}   # drop the candidate axis
    goal = model.encode(goal, target='goal_emb', pixels_key='goal', prefix='goal_', emb_keys=emb_keys)
    for key in ('goal_emb', 'pixels_goal_emb', *(f'{k}_goal_emb' for k in emb_keys)):
        info_dict[key] = goal[key]
    return info_dict['goal_emb']


def _splits_latent(model) -> bool:
    "Whether `model` fuses extra sources into a split latent (its `encode` takes `emb_keys`)."
    if not getattr(model, 'extra_encoders', None):
        return False
    import inspect
    try:
        return 'emb_keys' in inspect.signature(model.encode).parameters
    except (TypeError, ValueError):
        return False


def default_goal_encode(model: Dynamics, info_dict: dict) -> torch.Tensor:
    """
    The goal encoding matching the model's latent: `split_goal_encode` for models with a
    split latent (extra encoders, and an `encode` taking `emb_keys`), else
    `flat_goal_encode` with each agent's `pixels`. stable-worldmodel dispatches on
    `extra_encoders` alone; here LeWM's extra encoders are projected into one flat latent, so
    it keeps the flat path.
    """
    if _splits_latent(model):
        return split_goal_encode(model, info_dict)
    return flat_goal_encode(model, info_dict)

The model's `encode(info, pixels_key=, emb_keys=, prefix=, target=)` reads the goal-side
inputs (`goal`, `goal_<key>`) and writes `<target>`, `pixels_<target>`,
`<key>_<target>`.

In [ ]:
#| export
class ShootingCostEvaluator(torch.nn.Module):
    """
    Single-shooting adapter making `(model, objective)` a `Costable`: it encodes the
    goal (once), rolls the candidate action sequences out with the model in one shot, then scores
    the predicted trajectories with the objective. Solvers consume it like a world model.
    """
    def __init__(
        self,
        model: Dynamics, # World model with `encode` / `rollout` (registered as a submodule)
        objective: Objective, # Cost of a rolled-out info dict: `(B, S)` joint or `(B, S, num_agents)` per agent
        constraints: list[Objective] | None = None, # Terms used as constraints (satisfied when `<= 0`), exposed by `get_constraints`
        encode_goal: Callable[[Dynamics, dict], torch.Tensor] | None = default_goal_encode # `fn(model, info_dict) -> goal_emb` (None: the caller provides `goal_emb`)
    ):
        super().__init__()
        self.model, self.objective, self.constraints, self.encode_goal = model, objective, constraints, encode_goal
        if constraints:
            self.get_constraints = self._get_constraints

    def _rollout(self, info_dict: dict, action_candidates: torch.Tensor) -> dict:
        if self.encode_goal is not None and 'goal_emb' not in info_dict:
            info_dict['goal_emb'] = self.encode_goal(self.model, info_dict)
        info_dict = self.model.rollout(info_dict, action_candidates)
        info_dict['action_candidates'] = action_candidates
        return info_dict

    def criterion(self, info_dict: dict, action_candidates: torch.Tensor | None = None) -> torch.Tensor:
        "Score an already rolled-out info dict."
        if action_candidates is not None:
            info_dict['action_candidates'] = action_candidates
        return self.objective(info_dict)

    def get_cost(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        "Encode the goal (if needed), roll the candidates out, then score them."
        return self.objective(self._rollout(info_dict, action_candidates))

    def _get_constraints(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        info_dict = self._rollout(info_dict, action_candidates)
        return torch.stack([term(info_dict) for term in self.constraints], dim=-1)

In [ ]:
show_doc(ShootingCostEvaluator.criterion)

In [ ]:
show_doc(ShootingCostEvaluator.get_cost)

In [ ]:
# split latent (PreJEPA-like): per-source goal embeddings are stored, the fused one returned
class _Split(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.extra_encoders = torch.nn.ModuleDict({'position': torch.nn.Linear(2, 3), 'action': torch.nn.Linear(1, 3)})
    def encode(self, info, pixels_key='pixels', emb_keys=None, prefix=None, target='emb'):
        prefix, emb_keys = prefix or '', list(self.extra_encoders) if emb_keys is None else emb_keys
        info[f'pixels_{target}'] = info[pixels_key].flatten(2).float().mean(-1, keepdim=True)
        parts = [info[f'pixels_{target}']]
        for k in emb_keys:
            info[f'{k}_{target}'] = self.extra_encoders[k](info[f'{prefix}{k}'].float())
            parts.append(info[f'{k}_{target}'])
        info[target] = torch.cat(parts, -1)
        return info

# (batch, candidate, time, ...): the goal-side inputs of 2 envs, 3 frames
info = {'goal': torch.rand(2, 1, 3, 8, 8), 'goal_position': torch.rand(2, 1, 3, 2), 'action': torch.rand(2, 1, 3, 1)}
emb = default_goal_encode(_Split(), info)
assert emb.shape == (2, 3, 4) and info['pixels_goal_emb'].shape == (2, 3, 1) and info['position_goal_emb'].shape == (2, 3, 3)
assert 'action_goal_emb' not in info and torch.equal(info['goal_emb'], emb)

# LeWM projects its extra encoders into one flat latent: the flat path
from stable_marl.wm import build_lewm
wm = build_lewm(action_dim=5, image_size=16, patch_size=8, depth=1, heads=2, dim_head=8, mlp_dim=32, projector_hidden=32,
                encoder_kwargs={'dim': 32, 'depth': 1, 'heads': 2, 'mlp_dim': 32}, extra_inputs={'position': 2})
assert wm.extra_encoders and not _splits_latent(wm)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()